# Fixed-Number Image Augmentation

Uses the [Augmentor](https://github.com/mdbloice/Augmentor) library to
generate a fixed number of augmented samples per class via random rotation,
zoom, and horizontal/vertical flipping. This balances class distribution
across folds before training.

Two usage patterns are provided:
1. **Single folder** — augment one class folder to a target count.
2. **All folds** — loop through every fold/class folder under a base path.

Update all `<PLACEHOLDER>` paths before running.


## Setup

In [ ]:
!pip install Pillow Augmentor

In [ ]:
import os
import shutil
import Augmentor

## 1. Augment a Single Folder

Copies original images into the output folder, then augments them up to
`num_samples` total images.

In [ ]:
def augment_single_folder(input_folder, output_folder, num_samples):
    """
    Copies images from input_folder into output_folder, then generates
    augmented samples (rotation, zoom, flipping) until num_samples total
    images exist in output_folder.
    """
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)

    # Copy original images into the output folder first
    for img_name in os.listdir(input_folder):
        src_path = os.path.join(input_folder, img_name)
        dst_path = os.path.join(output_folder, img_name)
        shutil.copyfile(src_path, dst_path)

    # Build augmentation pipeline
    pipeline = Augmentor.Pipeline(output_folder)
    pipeline.rotate(probability=1, max_left_rotation=10, max_right_rotation=10)
    pipeline.zoom_random(probability=1, percentage_area=0.8)
    pipeline.flip_left_right(probability=1)
    pipeline.flip_top_bottom(probability=1)

    pipeline.sample(num_samples)

In [ ]:
input_folder = "<INPUT_FOLDER>"
output_folder = "<OUTPUT_FOLDER>"
num_samples = 24909  # target total image count

augment_single_folder(input_folder, output_folder, num_samples)

## 2. Augment All Folds and Classes

Loops through every fold (`fold_1` ... `fold_n`) and every class folder
inside each fold, augmenting images in place up to `num_samples` per
class.

In [ ]:
def augment_images_in_place(folder_path, num_samples):
    """
    Augments images directly inside folder_path (no copy step) until
    num_samples total images exist.
    """
    pipeline = Augmentor.Pipeline(source_directory=folder_path)
    pipeline.rotate(probability=1, max_left_rotation=10, max_right_rotation=10)
    pipeline.zoom_random(probability=1, percentage_area=0.8)
    pipeline.flip_left_right(probability=1)
    pipeline.flip_top_bottom(probability=1)

    pipeline.sample(num_samples)


def process_all_folds(base_path, num_folds=5, num_samples=3000):
    for fold in range(1, num_folds + 1):
        fold_path = os.path.join(base_path, f"fold_{fold}")
        for class_name in os.listdir(fold_path):
            class_path = os.path.join(fold_path, class_name)
            augment_images_in_place(class_path, num_samples=num_samples)

In [ ]:
base_path = r"<TRAIN_BASE_PATH>"

process_all_folds(base_path, num_folds=5, num_samples=3000)